# 🎬 Cloud AI Video Worker (Wan 2.1 / LTX-Video)
### Free Tesla T4 GPU + Persistent Storage + Cloudflare Tunnel

**⚠️ Before running — do this FIRST:**
> **1. Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**
> **2. Runtime → Run all**

When finished, copy the **trycloudflare.com** URL printed in Cell 4 and paste it into your Desktop App.

---

In [ ]:
#@title 1. ✅ Check GPU & Mount Storage
import subprocess, os, sys

# ── 1. Verify GPU is available BEFORE anything else ───────────────────────────
print('🔍 Checking GPU runtime...')
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=10)
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError('nvidia-smi returned empty')
    print(f'✅ GPU detected: {r.stdout.strip()}')
except Exception:
    print('\n' + '═'*65)
    print('❌  NO GPU DETECTED — Requires a T4 GPU runtime!')
    print('═'*65)
    print('\n👉 How to switch:')
    print('    Runtime → Change runtime type → T4 GPU → Save → Run all\n')
    raise SystemExit('No GPU. Switch runtime to T4 GPU first.')

# ── 2. Verify PyTorch CUDA ────────────────────────────────────────────────────
cuda_ok = subprocess.run(
    [sys.executable, '-c', 'import torch; assert torch.cuda.is_available(), "CUDA not available"'],
    capture_output=True, text=True
)
if cuda_ok.returncode != 0:
    print('\n' + '═'*65)
    print('❌  PyTorch CUDA not working!')
    print('═'*65)
    print('\n👉 Fix: Runtime → Factory reset runtime → Change type to T4 GPU → Run all\n')
    raise SystemExit('PyTorch CUDA unavailable.')

cuda_ver = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, "CUDA", torch.version.cuda)'],
                          capture_output=True, text=True).stdout.strip()
print(f'✅ PyTorch CUDA OK: {cuda_ver}')

# ── 3. Mount Google Drive (Optional persistence) ──────────────────────────────
DRIVE_BASE = '/content/AiVideoWorker'
try:
    from google.colab import drive  # type: ignore  # noqa
    print('\n🔗 Mounting Google Drive...')
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/AiVideoWorker'
    print(f'✅ Persistent Google Drive mounted at: {DRIVE_BASE}')
except Exception as e:
    print(f'⚠️ Google Drive mount skipped ({e}). Using temporary session storage: {DRIVE_BASE}')

for d in ['models/checkpoints', 'models/vae', 'models/clip', 'outputs']:
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)
print('📁 Storage directories ready.')

In [ ]:
#@title 2. 📦 Install ComfyUI & Video Extensions
import os, shutil

# Clone ComfyUI
if not os.path.exists("/content/ComfyUI"):
    print("📦 Cloning ComfyUI...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI

# Link models to persistent storage using robust Python path resolution
print("🔗 Linking model directories to ComfyUI...")
for model_dir in ["checkpoints", "vae", "clip", "text_encoders"]:
    target = f"/content/ComfyUI/models/{model_dir}"
    source = f"{DRIVE_BASE}/models/{model_dir if model_dir != 'text_encoders' else 'clip'}"
    os.makedirs(source, exist_ok=True)
    if os.path.islink(target) or os.path.exists(target):
        try:
            if os.path.islink(target) or os.path.isfile(target): os.unlink(target)
            else: shutil.rmtree(target)
        except Exception: pass
    os.symlink(source, target)
    print(f"  ✓ Linked /content/ComfyUI/models/{model_dir} -> {source}")

# ⚠️ DO NOT reinstall PyTorch! Colab has the correct CUDA PyTorch pre-installed.
print("\n⚙️ Installing ComfyUI dependencies...")
!pip install -q -r requirements.txt 2>&1 | grep -v "already satisfied" || true
!pip install -q diffusers accelerate safetensors sentencepiece huggingface_hub einops 2>&1 | grep -v "already satisfied" || true

# Install Video custom nodes
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("ComfyUI-VideoHelperSuite"):
    !git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite
    !pip install -q -r ComfyUI-VideoHelperSuite/requirements.txt 2>&1 | grep -v "already satisfied" || true

if not os.path.exists("ComfyUI-WanVideoWrapper"):
    !git clone --depth 1 https://github.com/kijai/ComfyUI-WanVideoWrapper
    !pip install -q -r ComfyUI-WanVideoWrapper/requirements.txt 2>&1 | grep -v "already satisfied" || true

if not os.path.exists("ComfyUI-LTXVideo"):
    !git clone --depth 1 https://github.com/city96/ComfyUI-LTXVideo || true

%cd /content/ComfyUI
print("\n✅ ComfyUI and video extensions ready!")


In [ ]:
#@title 3. 📥 Download AI Video Models (One-time download)
import os, subprocess, shutil

CHECKPOINTS = f"{DRIVE_BASE}/models/checkpoints"
CLIP        = f"{DRIVE_BASE}/models/clip"
VAE         = f"{DRIVE_BASE}/models/vae"
for d in [CHECKPOINTS, CLIP, VAE]: os.makedirs(d, exist_ok=True)

def download_if_missing(label, dest_path, url, min_mb=50):
    if os.path.exists(dest_path):
        mb = os.path.getsize(dest_path) / 1e6
        if mb >= min_mb:
            print(f"✅ {label} already present ({mb:.0f} MB)")
            return True
        else:
            print(f"⚠️  {label} exists but is incomplete ({mb:.1f} MB) — re-downloading...")
            try: os.remove(dest_path)
            except: pass

    print(f"📥 Downloading {label}...")
    # Use curl with location follow (-L) and resume (-C -)
    r = subprocess.run(["curl", "-L", "-C", "-", "--retry", "3", "-f", url, "-o", dest_path])
    if r.returncode != 0 or not os.path.exists(dest_path):
        print(f"⚠️ curl failed, trying wget fallback...")
        r = subprocess.run(["wget", "-q", "--show-progress", url, "-O", dest_path])
        if r.returncode != 0 or not os.path.exists(dest_path):
            print(f"❌ {label} download FAILED (exit code {r.returncode})")
            return False
    mb = os.path.getsize(dest_path) / 1e6
    if mb < min_mb:
        print(f"❌ {label} downloaded but only {mb:.1f} MB (likely 404 HTML). Deleting.")
        try: os.remove(dest_path)
        except: pass
        return False
    print(f"✅ {label} downloaded ({mb:.0f} MB)")
    return True

# 1. T5-XXL FP8 text encoder (~4.8 GB)
download_if_missing(
    "T5-XXL FP8 text encoder (~4.8 GB)",
    f"{CLIP}/t5xxl_fp8_e4m3fn.safetensors",
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
    min_mb=1000
)
# Also symlink into text_encoders for newer ComfyUI
try:
    os.makedirs(f"{DRIVE_BASE}/models/text_encoders", exist_ok=True)
    te_path = f"{DRIVE_BASE}/models/text_encoders/t5xxl_fp8_e4m3fn.safetensors"
    if not os.path.exists(te_path):
        os.symlink(f"{CLIP}/t5xxl_fp8_e4m3fn.safetensors", te_path)
except Exception: pass

# 1b. Wan 2.1 UMT5-XXL FP8 text encoder (~4.8 GB for Wan 2.1)
download_if_missing(
    "Wan 2.1 UMT5 text encoder (~4.8 GB)",
    f"{CLIP}/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    min_mb=1000
)
try:
    umt5_te = f"{DRIVE_BASE}/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"
    if not os.path.exists(umt5_te):
        os.symlink(f"{CLIP}/umt5_xxl_fp8_e4m3fn_scaled.safetensors", umt5_te)
except Exception: pass

# 2. Wan 2.1 VAE (~250 MB)
vae_bf16 = f"{VAE}/Wan2_1_VAE_bf16.safetensors"
vae_fp8  = f"{VAE}/Wan2_1_VAE_fp8.safetensors"
if download_if_missing(
    "Wan 2.1 VAE (~250 MB)",
    vae_bf16,
    "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1_VAE_bf16.safetensors",
    min_mb=100
):
    if not os.path.exists(vae_fp8):
        try: os.symlink(vae_bf16, vae_fp8)
        except: pass

# 3. LTX-Video 2B (~5.7 GB) — Primary super-fast Image-to-Video engine for T4
download_if_missing(
    "LTX-Video 2B (~5.7 GB)",
    f"{CHECKPOINTS}/ltx-video-2b-v0.9.1.safetensors",
    "https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.1.safetensors",
    min_mb=1000
)

# 4. Wan 2.1 1.3B (~1.4 GB)
wan_t2v = f"{CHECKPOINTS}/Wan2_1-T2V-1_3B_fp8_e4m3fn.safetensors"
wan_alias = f"{CHECKPOINTS}/wan2.1_i2v_1.3B_fp8.safetensors"
if download_if_missing(
    "Wan 2.1 1.3B (~1.4 GB)",
    wan_t2v,
    "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1-T2V-1_3B_fp8_e4m3fn.safetensors",
    min_mb=500
):
    if not os.path.exists(wan_alias):
        try: os.symlink(wan_t2v, wan_alias)
        except: pass

# Re-verify ComfyUI symlinks
for model_dir in ["checkpoints", "vae", "clip", "text_encoders"]:
    target = f"/content/ComfyUI/models/{model_dir}"
    source = f"{DRIVE_BASE}/models/{model_dir if model_dir != 'text_encoders' else 'clip'}"
    if not os.path.exists(target):
        try: os.symlink(source, target)
        except Exception: pass

# Inventory check
print("\n" + "═"*50)
print("📦 Model Inventory Check:")
for label, p in [("T5-XXL Text Encoder", f"{CLIP}/t5xxl_fp8_e4m3fn.safetensors"),
                  ("Wan 2.1 VAE", vae_bf16),
                  ("LTX-Video 2B", f"{CHECKPOINTS}/ltx-video-2b-v0.9.1.safetensors"),
                  ("Wan 2.1 1.3B", wan_t2v)]:
    if os.path.exists(p): print(f"  ✅ {label}: {os.path.getsize(p)/1e6:.0f} MB")
    else: print(f"  ❌ {label}: NOT FOUND")
print("═"*50)


In [ ]:
#@title 4. 🚀 Start ComfyUI + Cloudflare Tunnel — COPY THE URL TO YOUR APP
import subprocess, time, re, os

COMFY_LOG = '/content/comfy_server.log'

# ── 1. Download Cloudflared ─────────────────────────────────────────────────────
if not os.path.exists('/content/cloudflared'):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

# ── 2. Terminate any previous instances ─────────────────────────────────────────
!pkill -9 -f 'python main.py' 2>/dev/null || true
!pkill -9 -f 'cloudflared'   2>/dev/null || true
time.sleep(2)

# ── 3. Start ComfyUI ────────────────────────────────────────────────────────────
print('🚀 Starting ComfyUI server on port 8188...')
with open(COMFY_LOG, 'w') as f: pass
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
     '--enable-cors-header', '--preview-method', 'auto'],
    cwd='/content/ComfyUI',
    stdout=open(COMFY_LOG, 'w'),
    stderr=subprocess.STDOUT,
    text=True
)

# ── 4. Health-Check: Wait until ComfyUI responds on port 8188 ───────────────────
print('⏳ Waiting for ComfyUI to initialize (max 120s)...')
comfy_ready = False
for i in range(120):
    time.sleep(1)
    if comfy_proc.poll() is not None:
        log = open(COMFY_LOG).read()
        print('\n' + '═'*65)
        print('❌ ComfyUI CRASHED on startup! Error log:')
        print('═'*65)
        print(log[-3000:])
        raise SystemExit('ComfyUI process exited prematurely. See log above.')

    try:
        r = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}',
                             'http://127.0.0.1:8188/system_stats'],
                            capture_output=True, text=True, timeout=3)
        if r.stdout.strip() == '200':
            print(f'✅ ComfyUI is up and healthy on port 8188! ({i+1}s)')
            comfy_ready = True
            break
    except: pass

    if (i + 1) % 15 == 0:
        print(f'   Still starting ComfyUI... ({i+1}s)')

if not comfy_ready:
    log = open(COMFY_LOG).read()
    print('\n' + '═'*65)
    print('⚠️ ComfyUI did not respond with HTTP 200 within 120s. Last log:')
    print(log[-1500:])
    print('═'*65)

# ── 5. Start Cloudflare Tunnel ──────────────────────────────────────────────────
print('\n🌐 Launching Cloudflare tunnel to ComfyUI...')
tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

if tunnel_url:
    print('\n' + '═'*70)
    print('🎉 YOUR CLOUD GPU TUNNEL URL:')
    print(f'👉  {tunnel_url}  👈')
    print('═'*70)
    print('\n📋 Next step:')
    print('  1. Copy the URL above.')
    print('  2. In your desktop app: Cloud AI Video Studio → Paste into Tunnel URL → Test & Connect.')
    print('  3. Keep this Colab tab OPEN while generating videos.\n')

    # Save to Drive if mounted
    try:
        if os.path.exists(DRIVE_BASE):
            with open(f'{DRIVE_BASE}/current_tunnel.txt', 'w') as f:
                f.write(tunnel_url)
            print(f'💾 Saved to {DRIVE_BASE}/current_tunnel.txt')
    except Exception as e:
        pass

    # Self-test through Cloudflare
    print('🔍 Verifying tunnel connectivity...')
    time.sleep(4)
    r = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}',
                         f'{tunnel_url}/system_stats'],
                        capture_output=True, text=True, timeout=15)
    code = r.stdout.strip()
    if code == '200':
        print('✅ Tunnel is LIVE and ComfyUI is responding (HTTP 200)! Ready to connect.')
    else:
        print(f'ℹ️ Tunnel HTTP status: {code} (if 502, wait ~15 seconds and test again in app).')
else:
    print('❌ Failed to capture trycloudflare.com URL from cloudflared.')

# ── 6. Keep-Alive loop ──────────────────────────────────────────────────────────
print('\n🟢 Server running. Keep this Colab tab active.')
try:
    while True:
        time.sleep(15)
        if comfy_proc.poll() is not None:
            print('⚠️ ComfyUI process stopped. Restarting...')
            comfy_proc = subprocess.Popen(
                ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
                 '--enable-cors-header', '--preview-method', 'auto'],
                cwd='/content/ComfyUI',
                stdout=open(COMFY_LOG, 'a'), stderr=subprocess.STDOUT, text=True
            )
            time.sleep(20)
        if tunnel_proc.poll() is not None:
            print('⚠️ Tunnel process stopped. Re-run Cell 4.')
            break
except KeyboardInterrupt:
    comfy_proc.terminate()
    tunnel_proc.terminate()
    print('Stopped.')

In [ ]:
#@title 5. 🛡️ Keep-Alive (Prevents Colab Idle Disconnect)
try:
    from IPython.display import display, Javascript  # type: ignore  # noqa
    display(Javascript('setInterval(() => { var b = document.querySelector("#top-toolbar > colab-connect-button"); if(b) b.click(); }, 60000);'))
    print('🟢 Browser keep-alive ping active (clicks connect button every 60s).')
except Exception as e:
    print(f'Keep-alive info: {e}')